In [ ]:
import pandas as pd
import numpy as np

In [2]:


# Load data
# 


FILE = "datasets/KIA 202604 Accessories Application Chart.xlsx"

xls = pd.ExcelFile(FILE)
print(xls.sheet_names)
 

['English List', 'To KCA (2)', 'Notes', 'Sell Down', 'TuxMat Catalog', 'Remote Starters', 'Wiper', 'Fastco Wheels-Roues Fastco', 'Liste française', 'Liquidation Accessoires', 'Template', 'Accy List', 'Q3 Price change list', 'Wiper Data', 'To KCA']


In [3]:
# load data from the sheet named "ACCY_list"
raw_accy_df = pd.read_excel(xls, sheet_name="Accy List")


In [4]:
def clean_column_names(df):
    df = df.copy()  # don't change raw_accy_df
    df.columns = (
        df.columns.str.replace("\n", " ")       # line break -> space, so words don't join
                  .str.strip()
                  .str.lower()
                  .str.replace(r"\s+", "_", regex=True)  # spaces -> _   <-- the missing step
                  .str.replace("_/_", "_")      # commentaire_/_remarques -> commentaire_remarques
                  .str.replace(r"_+", "_", regex=True)
    )
    return df

In [5]:
# Create a func that drops any empty columns from the dataframe and returns the cleaned dataframe. Then at the end, I want you to write another line that removes any columns that have all NaN values.

def drop_empty_columns(df):
    df = df.dropna(axis=1, how='all')
    df = df.dropna(axis=0, how='all')
    return df

 

In [6]:

raw_accy_df_cols_clean = clean_column_names(raw_accy_df)
raw_accy_df_cols_clean = drop_empty_columns(raw_accy_df_cols_clean)
 

In [7]:
raw_accy_df_cols_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2462 entries, 0 to 2461
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   model_year_from              2462 non-null   int64  
 1   model_year_to                2462 non-null   int64  
 2   model                        2462 non-null   str    
 3   trim_1                       2242 non-null   str    
 4   trim_2                       2313 non-null   str    
 5   trim_3                       1862 non-null   str    
 6   trim_4                       1334 non-null   str    
 7   trim_5                       1018 non-null   str    
 8   trim_6                       636 non-null    str    
 9   trim_7                       63 non-null     str    
 10  part_number                  2462 non-null   object 
 11  accessory_category_(en)      2462 non-null   str    
 12  accessory_group_(en)         2462 non-null   str    
 13  accessory_description_(en)   

In [8]:
essential_cols = {
    "part_description_cols" : ['part_number', 
                                'accessory_description_(en)', 
                                'comments/notes_(en)', 'accessory_description_(fr)', 'commentaire_remarques_(fr)'],
    "vehicle_description_cols" : ['model_year_from', 'model_year_to', 'model',],
    "trim_applicability_cols" : ['trim_1', 'trim_2', 'trim_3', 'trim_4', 'trim_5', 'trim_6', 'trim_7'], 
    "sales_data_cols": ['dealer_cost', 'suggested_base_retail_price', 'suggested_labour_hours']
}

In [9]:
# filter columns to essential columns only based on the dictionary above essential_cols based on all the keys in the dictionary. Then return a new dataframe with only the essential columns.

essential_columns = [col for cols in essential_cols.values() for col in cols]
raw_data_filtered = raw_accy_df_cols_clean[essential_columns]



In [10]:

raw_data_filtered.columns
 

Index(['part_number', 'accessory_description_(en)', 'comments/notes_(en)',
       'accessory_description_(fr)', 'commentaire_remarques_(fr)',
       'model_year_from', 'model_year_to', 'model', 'trim_1', 'trim_2',
       'trim_3', 'trim_4', 'trim_5', 'trim_6', 'trim_7', 'dealer_cost',
       'suggested_base_retail_price', 'suggested_labour_hours'],
      dtype='str')

In [11]:

raw_data_filtered.head()


,part_number,accessory_description_(en),comments/notes_(en),accessory_description_(fr),commentaire_remarques_(fr),model_year_from,model_year_to,model,trim_1,trim_2,trim_3,trim_4,trim_5,trim_6,trim_7,dealer_cost,suggested_base_retail_price,suggested_labour_hours
0,L3F03AP000,ACCESSORY HITCH,NOT COMPATIBLE WITH GT 2.5T TRIM,ACCESSOIRES DE TOIT,NE PAS COMPATIBLE AVEC GT 2.5T,2021,2024,K5,LX,EX,GT-Line,NaN,NaN,NaN,NaN,22.92,26.96,0.5
1,L3F12AC400,CARGO TRAY,NaN,PLATEAU D’ESPACE À BAGAGES,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,87.57,138.27,0.0
2,L3F05AK000,SPORT PEDALS (A/T),NaN,PÉDALE SPORT - AUTOMATIQUE,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,54.52,71.40,0.5
3,L3F17AP001,ALL WEATHER LINER FRONT (NEW LOGO),NaN,TOUTES SAISONS LINERS - AVANT (NOUVEAU LOGO),NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,57.28,69.61,0.0
4,L3F17AP101,ALL WEATHER LINER REAR (NEW LOGO),NaN,TOUTES SAISONS LINERS - ARRIÈRE (NOUVEAU LOGO),NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,27.38,32.21,0.0


In [12]:

col_mapping = {
    'model_year_from': ('model_year_from', int),
    'model_year_to': ('model_year_to', int),
    'model': ('model', str),
    'partnumber': ('partnumber', str),
    'accessory_description_(en)': ('description_en', str),
    'comments/notes_(en)': ('comments_en', str),
    'accessory_description_(fr)': ('description_fr', str),
    'commentaire_remarques_(fr)': ('comments_fr', str),
    'trim_1': ('trim_1', str),
    'trim_2': ('trim_2', str),
    'trim_3': ('trim_3', str),
    'trim_4': ('trim_4', str),
    'trim_5': ('trim_5', str),
    'trim_6': ('trim_6', str),
    'trim_7': ('trim_7', str),
    'dealer_cost': ('dealer_cost', float),
    'suggested_base_retail_price': ('MSRP', float),
    'suggested_labour_hours': ('Labour_hours', float),
}
 

In [13]:

# Function that takes in a df and the col_mapping dictionary and renames the columns based on the mapping, and also converts the data types of the columns based on the mapping. Then return the cleaned dataframe.

def rename_and_convert_data_types(df, col_mapping):
    # Rename columns based on the mapping
    df = df.rename(columns={k: v[0] for k, v in col_mapping.items()})
    
    # Convert data types based on the mapping
    for col, (new_name, dtype) in col_mapping.items():
        if col in df.columns:
            df[new_name] = df[col].astype(dtype)
    
    return df



In [14]:
raw_data_filtered_converted_types = rename_and_convert_data_types(raw_data_filtered, col_mapping)


In [76]:

raw_cleaned_df = raw_data_filtered_converted_types.copy()


In [77]:
# remove duplicated records, comparing the values in all the columns

def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"Removed {before - len(df)} duplicate rows ({before} -> {len(df)})")
    return df

raw_cleaned_df = remove_duplicates(raw_cleaned_df)


Removed 0 duplicate rows (2462 -> 2462)


In [78]:
import re
def find_parts(df, part_numbers, column='part_number', partial=False):
    """Return the rows whose part number matches.

    part_numbers: one part number ("L3F03AP000") or a list of them.
    partial=True also matches rows that contain the text (e.g. "L3F17" finds L3F17AP001 and L3F17AP101).
    Matching ignores case and surrounding spaces; part numbers are compared as given, never padded.
    """
    if isinstance(part_numbers, str):
        part_numbers = [part_numbers]
    wanted = [str(p).strip().upper() for p in part_numbers]

    # A column listed twice makes df[column] return a table: keep the first copy
    df = df.loc[:, ~df.columns.duplicated()]
    values = df[column].astype(str).str.strip().str.upper()

    if partial:
        pattern = '|'.join(map(re.escape, wanted))
        mask = values.str.contains(pattern, regex=True, na=False)
    else:
        mask = values.isin(wanted)

    result = df[mask]
    not_found = [p for p in wanted if not values.str.contains(re.escape(p), regex=True).any()] if partial \
                else sorted(set(wanted) - set(values[mask]))
    if not_found:
        print(f"Not found: {not_found}")
    print(f"{len(result)} rows for {len(wanted) - len(not_found)} of {len(wanted)} part numbers")
    return result

In [79]:
def first_labour_record(df, part_numbers, part_col='part_number', labour_col='Labour_hours'):
    """For each part number, the first record with a value in labour_col.

    Returns one row per part number, in the order given:
      found=True  -> that part's first record with labour hours (all its columns)
      found=False -> a message saying no record has labour hours (or the part isn't in df)
    Part numbers are matched exactly (ignoring case and spaces), never padded.
    """
    if isinstance(part_numbers, str):
        part_numbers = [part_numbers]

    df = df.loc[:, ~df.columns.duplicated()]       # a duplicated column would make df[col] a table
    keys = df[part_col].astype(str).str.strip().str.upper()
    labour = pd.to_numeric(df[labour_col], errors='coerce')   # "nan" text / blanks -> missing

    rows = []
    for pn in dict.fromkeys(str(p).strip().upper() for p in part_numbers):   # unique, order kept
        records = df[keys == pn]
        with_labour = records[labour.loc[records.index].notna()]

        if records.empty:
            rows.append({'lookup_part_number': pn, 'found': False,
                         'message': 'Part number not in the data'})
        elif with_labour.empty:
            rows.append({'lookup_part_number': pn, 'found': False,
                         'message': f'No record with {labour_col} ({len(records)} records, all blank)'})
        else:
            first = with_labour.iloc[0]
            rows.append({'lookup_part_number': pn, 'found': True,
                         'message': f'{labour_col} = {first[labour_col]} '
                                    f'(row {first.name}, {len(with_labour)} of {len(records)} records have it)',
                         'source_row': first.name, **first.to_dict()})

    result = pd.DataFrame(rows)
    print(f"{int(result['found'].sum())} of {len(result)} part numbers have a {labour_col} value")
    return result

## Exploratory Data Analysis

In [80]:
# Year coverage in the dataset

# Create a function that takes in a dataframe and returns the minimum and maximum year from the "model_year_from" and "model_year_to" columns. Then return the min and max year as a tuple.
def get_year_coverage(df):
    df = df.loc[:, ~df.columns.duplicated()]
    min_year = df['model_year_from'].min()
    max_year = df['model_year_to'].max()
    return (min_year, max_year)


In [81]:
get_year_coverage(raw_cleaned_df)



(np.int64(2018), np.int64(2026))

In [82]:
# Understanding models represented in the dataset

def get_unique_models_and_counts(df):
    model_counts = df['model'].value_counts()
    return model_counts

In [83]:
get_unique_models_and_counts(raw_cleaned_df)

model
Niro EV             152
EV6                 140
Sorento              94
Sorento PE           91
Telluride            89
Carnival PE          88
Sorento PHEV         86
Sorento PHEV PE      86
Carnival             86
Sorento HEV          84
Sorento HEV PE       84
Carnival HEV         84
EV9                  84
Seltos               83
Sportage             82
Sportage PHEV        79
Sportage PHEV PE     78
Sportage PE          77
Niro HEV             77
Sportage HEV         76
Sportage HEV PE      75
K5                   74
Niro PHEV            73
K4 Hatchback         65
K4                   64
EV6 PE               49
Rio/Rio5             48
Forte 5 PE           42
Forte PE             40
EV4                  40
Soul                 39
Soul EV              34
Stinger              19
Name: count, dtype: int64

In [84]:
# Reviewing the numeric values

# Create a function that takes in a dataframe, a list of columns, and for each column it it will print the column name, the minimum value, the maximum value, the mean, and the standard deviation. Then return a new dataframe with the summary statistics for each column.
def analyze_numeric_columns(df, columns):
    summary_stats = []
    for col in columns:
        # print(f"Column: {col}")
        # print(f"Minimum: {df[col].min()}")
        # print(f"Maximum: {df[col].max()}")
        # print(f"Mean: {df[col].mean()}")
        # print(f"Standard Deviation: {df[col].std()}")
        summary_stats.append({
            'Column': col,
            'Min_val': df[col].min(),
            'Max_val': df[col].max(),
            'Mean': df[col].mean(),
            'Standard Deviation': df[col].std()
        })
    return pd.DataFrame(summary_stats)
 

In [85]:
raw_cleaned_df.columns

Index(['part_number', 'description_en', 'comments_en', 'description_fr',
       'comments_fr', 'model_year_from', 'model_year_to', 'model', 'trim_1',
       'trim_2', 'trim_3', 'trim_4', 'trim_5', 'trim_6', 'trim_7',
       'dealer_cost', 'MSRP', 'Labour_hours'],
      dtype='str')

In [86]:

analyze_numeric_columns(raw_cleaned_df, ['dealer_cost', 'MSRP','Labour_hours'])
 

,Column,Min_val,Max_val,Mean,Standard Deviation
0,dealer_cost,1.56,1133.91,213.785093,229.591897
1,MSRP,1.83,1889.85,301.643323,325.034953
2,Labour_hours,0.00,4.00,0.160014,0.307724


In [87]:
# business rules for the data quality check (names are the renamed ones from col_mapping)
data_rules = {
    'greater_than':     [('MSRP', 'dealer_cost')],                 # left value must be > right value
    'greater_or_equal': [('model_year_to', 'model_year_from')],    # left value must be >= right value
    'not_null': ['model_year_from', 'model_year_to', 'model', 'part_number',
                 'accessory_description_(en)', 'dealer_cost', 'suggested_base_retail_price', 'suggested_labour_hours'],
    'any_not_null':     [['trim_1', 'trim_2', 'trim_3', 'trim_4', 'trim_5', 'trim_6', 'trim_7']],  # at least one per row
    'numeric_non_negative': ['dealer_cost', 'MSRP', 'Labour_hours'],  # zero or more
}

In [88]:
# business rules for the data quality check
# Names can be old (source) or new: old names are converted to new ones through col_mapping
data_rules = {
    'greater_than':     [('MSRP', 'dealer_cost')],                 # left value must be > right value
    # 'greater_or_equal': [('model_year_to', 'model_year_from')],    # left value must be >= right value
    'not_null': ['model_year_from', 'model_year_to', 'model', 'part_number',
                 'description_en', 'dealer_cost', 'MSRP', 'Labour_hours'],
    'any_not_null':     [['trim_1', 'trim_2', 'trim_3', 'trim_4', 'trim_5', 'trim_6', 'trim_7']],  # at least one per row
    'numeric_non_zero': ['dealer_cost', 'MSRP', 'model_year_from', 'model_year_to'],
    'numeric_non_negative': ['Labour_hours'],                      # zero or more
}


def data_quality_check(df, data_rules, column_mapping=col_mapping):
    """Check df against data_rules. Returns (report, failed_rows).

    report:      one row per check (rule, columns, PASS / FAIL / MISSING COLUMN, failed row count)
    failed_rows: {"rule: columns": rows that broke it}
    """
    rename = {src: new for src, (new, _) in column_mapping.items()}
    col = lambda c: rename.get(c, c)   # old name -> new name; new names stay as they are

    # A column listed twice (e.g. partnumber) makes df[c] return a table, not a column: keep the first
    dupes = df.columns[df.columns.duplicated()].unique().tolist()
    if dupes:
        print(f"Note: duplicate columns kept once: {dupes}")
        df = df.loc[:, ~df.columns.duplicated()]

    results = []      # one row per rule check
    failed_parts = []  # failed rows from each check; combined into one DataFrame at the end  # rule description -> rows that broke it

    def record(rule, columns, mask=None, note=''):
        n_failed = int(mask.sum()) if mask is not None else None
        status = 'MISSING COLUMN' if mask is None else ('PASS' if n_failed == 0 else 'FAIL')
        results.append({'rule': rule, 'columns': columns, 'status': status,
                        'failed_rows': n_failed, 'note': note})
        if mask is not None and n_failed:
            failed_parts.append(
                df[mask].assign(dq_rule=rule, dq_columns=columns)
                        .rename_axis('source_row').reset_index()
            )

    def missing(*cols):
        return [c for c in cols if c not in df.columns]

    def blank(c):
        # Missing value, empty text, or "nan" text left over from .astype(str)
        s = df[c]
        return s.isna() | s.astype(str).str.strip().str.lower().isin(['', 'nan', 'none'])

    # Comparisons between two columns
    for rule, op, compare in (('greater_than', '>', lambda l, r: l > r),
                              ('greater_or_equal', '>=', lambda l, r: l >= r)):
        for left, right in data_rules.get(rule, []):
            l, r = col(left), col(right)
            label = f"{l} {op} {r}"
            if missing(l, r):
                record(rule, label, note=f"not in df: {missing(l, r)}")
                continue
            ok = compare(pd.to_numeric(df[l], errors='coerce'),
                         pd.to_numeric(df[r], errors='coerce'))
            record(rule, label, ~ok.fillna(False).astype(bool))

    # Each listed column must have a value on every row
    for c in map(col, data_rules.get('not_null', [])):
        if missing(c):
            record('not_null', c, note=f"'{c}' not in df")
            continue
        record('not_null', c, blank(c))

    # At least one column in each group must have a value (trims: else the accessory is orphaned)
    for group in data_rules.get('any_not_null', []):
        group = [col(c) for c in group]
        label = ', '.join(group)
        present = [c for c in group if c in df.columns]
        if not present:
            record('any_not_null', label, note=f"none of {group} in df")
            continue
        all_blank = pd.concat([blank(c) for c in present], axis=1).all(axis=1)
        note = f"not in df: {missing(*group)}" if missing(*group) else ''
        record('any_not_null', label, all_blank, note=note)

    # Numeric rules: a value fails if it isn't a number, or if it breaks the condition
    numeric_fails = {
        'numeric_non_zero':     lambda v: v == 0,
        'numeric_non_negative': lambda v: v < 0,
    }
    for rule, fails in numeric_fails.items():
        for c in map(col, data_rules.get(rule, [])):
            if missing(c):
                record(rule, c, note=f"'{c}' not in df")
                continue
            values = pd.to_numeric(df[c], errors='coerce')
            record(rule, c, (values.isna() | fails(values)).astype(bool))

    
    report = pd.DataFrame(results)
    failed_rows = (pd.concat(failed_parts, ignore_index=True) if failed_parts
                   else pd.DataFrame(columns=['source_row', 'dq_rule', 'dq_columns', *df.columns]))
    # Rule columns first, then the data
    failed_rows = failed_rows[['source_row', 'dq_rule', 'dq_columns',
                               *[c for c in failed_rows.columns if c not in ('source_row', 'dq_rule', 'dq_columns')]]]

    broken = report['status'].ne('PASS').sum()
    print(f"Data quality check: {len(report)} checks, {broken} with issues, "
          f"{failed_rows['source_row'].nunique()} rows affected")
    return report, failed_rows

data_quality_report, failed_rows = data_quality_check(raw_cleaned_df, data_rules)

Data quality check: 15 checks, 3 with issues, 318 rows affected


In [89]:

data_quality_report
 

,rule,columns,status,failed_rows,note
0,greater_than,MSRP > dealer_cost,PASS,0,
1,not_null,model_year_from,PASS,0,
2,not_null,model_year_to,PASS,0,
3,not_null,model,PASS,0,
4,not_null,part_number,PASS,0,
5,not_null,description_en,PASS,0,
6,not_null,dealer_cost,PASS,0,
7,not_null,MSRP,PASS,0,
8,not_null,Labour_hours,FAIL,310,
9,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",FAIL,8,


In [90]:
# Parts missing trims applicability
# Noticed that some are missing part# (TBD)
# 4 part numbers are missing applicability "trims"

''' 

Action:
- how these data point should be interpreted
- what should be done when we get this kind of data in the future


Note:
- In-development parts: Some records say "in development", need to know exactly how to handle these. Ideally, these are accessories that are still in dev, they will come out in teh future. Since they don't have part# we can't fetch in the part_master. 
- Un sure if they apply to all : Need to confirm if not having applicability could mean parts apply to all teh vehicles in the model family.

'''

failed_rows[
    (failed_rows["dq_rule"]=="any_not_null")
    & (failed_rows["dq_columns"].str.contains("trim_"))
    ]["part_number"].value_counts()
 

part_number
DWH46AP000     4
TBD            2
000AM481004    1
9999611001     1
Name: count, dtype: int64

In [91]:
failed_rows[
    (failed_rows["dq_rule"]=="any_not_null")
    & (failed_rows["dq_columns"].str.contains("trim_"))
    ]

,source_row,dq_rule,dq_columns,part_number,description_en,comments_en,description_fr,comments_fr,model_year_from,model_year_to,...,trim_1,trim_2,trim_3,trim_4,trim_5,trim_6,trim_7,dealer_cost,MSRP,Labour_hours
310,1029,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2026,2026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
311,1104,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2023,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
312,1179,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2026,2026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
313,1257,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2023,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
314,2271,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",TBD,CARGO TRAY,In development,PLATEAU DE CARGAISON,En développement,2026,2026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.99,139.99,0.0
315,2291,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",TBD,MUD FLAPS BY RALLY ARMOR,In development,GARDE-BOUE RALLY ARMOR,En développement,2026,2026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,268.01,382.87,0.5
316,2405,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",000AM481004,PREMIUM BLOCK HEATER KIT (2.0),NaN,ENSEMBLE CHAUFFE MOTEUR DE LUXE (2.0),NaN,2019,2023,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,132.62,205.61,1.0
317,2412,any_not_null,"trim_1, trim_2, trim_3, trim_4, trim_5, trim_6...",9999611001,WHEEL LOCKS,NaN,VERROUS DE ROUE,NaN,2018,2019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.26,70.59,0.2


In [92]:
'''
Found part# missing applicability. 

'''
find_parts(raw_cleaned_df, 'DWH46AP000', partial=True)
 

4 rows for 1 of 1 part numbers


,part_number,description_en,comments_en,description_fr,comments_fr,model_year_from,model_year_to,model,trim_1,trim_2,trim_3,trim_4,trim_5,trim_6,trim_7,dealer_cost,MSRP,Labour_hours
1029,DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2026,2026,Sportage HEV PE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
1104,DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2023,2025,Sportage HEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
1179,DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2026,2026,Sportage PHEV PE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5
1257,DWH46AP000,MUD FLAPS BY RALLY ARMOR,NaN,GARDE-BOUE - RALLY ARMOR,NaN,2023,2025,Sportage PHEV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.29,266.13,0.5


In [93]:
'''

Labour Hours:

'''


labour_hours_non_num_issues = failed_rows[
    (failed_rows["dq_rule"] == "numeric_non_negative")
    & (failed_rows["dq_columns"] == "Labour_hours")
] 

In [94]:
# Records missing Labour hours
labour_hours_non_num_issues["Labour_hours"].value_counts(dropna=False)
 

Labour_hours
NaN    310
Name: count, dtype: int64

In [95]:
# failed_rows[
#     (failed_rows["dq_rule"] == "numeric_non_negative")
#     & (failed_rows["dq_columns"] == "Labour_hours")
# ]

In [ ]:
# checking to see if records with empty labour hours have something in other records using partnumber as a unique key. This would help to complete that records, that if the assumption is valid.

parts = labour_hours_non_num_issues["part_number"].unique()
labour_lookup = first_labour_record(raw_cleaned_df, parts)

labour_lookup[['lookup_part_number', 'found', 'message']]   # quick view
labour_lookup[labour_lookup['found']]                       # full first record for the parts that have hours
 

3 of 20 part numbers have a Labour_hours value


,lookup_part_number,found,message,source_row,part_number,description_en,comments_en,description_fr,comments_fr,model_year_from,...,trim_1,trim_2,trim_3,trim_4,trim_5,trim_6,trim_7,dealer_cost,MSRP,Labour_hours
4,000AK18MP02,True,"Labour_hours = 0.2 (row 2235, 1 of 27 records ...",2235.0,000AK18MP02,"ALLOY WHEEL - 18""",NaN,JANTES EN ALLIAGE (18),NaN,2026.0,...,Light FWD,Wind FWD,Wind Premium FWD,GTL FWD,GTL Premium FWD,NaN,NaN,261.69,348.92,0.2
18,CVF40AK020,True,"Labour_hours = 0.3 (row 2014, 2 of 3 records h...",2014.0,CVF40AK020,"ALLOY WHEEL 19"" (SR, LR AND GT-LINE ONLY)",NaN,"JANTES EN ALLIAGE (19) NOIR (SR, LR AND GT-LIN...",NaN,2022.0,...,SR RWD,LR RWD,LR AWD,GT-Line,NaN,NaN,NaN,251.59,317.79,0.3
19,BEH40AKA00,True,"Labour_hours = 0.0 (row 2139, 1 of 3 records h...",2139.0,BEH40AKA00,TPMS KIT 4 PACK,NaN,TPMS KIT 4 PACK,NaN,2025.0,...,Light RWD,Wind RWD,Land AWD,Land AWD Premium Pkg,Land AWD GT-Line Pkg,Land AWD GT-Line Limited Pkg,NaN,70.26,117.10,0.0


In [ ]:
# Remaining part# w/o same part number records with labour hours to fill with. 

'''
Need to determine what to do with these parts.

'''

find_parts( raw_cleaned_df, 
           labour_lookup[labour_lookup["found"]==False]["lookup_part_number"].unique().tolist()).drop_duplicates(subset="part_number")
 

281 rows for 17 of 17 part numbers


,part_number,description_en,comments_en,description_fr,comments_fr,model_year_from,model_year_to,model,trim_1,trim_2,trim_3,trim_4,trim_5,trim_6,trim_7,dealer_cost,MSRP,Labour_hours
29,000AM472010,WARMUP KIT 1200W,NaN,ENSEMBLE DE PRÉCHAUFFAGE,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,474.90,736.16,NaN
30,000AM472007,WARMUP KIT 1200W - BT,NaN,ENSEMBLE DE PRÉCHAUFFAGE - BLUETOOTH,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,549.94,852.36,NaN
53,000AK17MP01,ALLOY WHEEL 17 x 7,NaN,JANTES EN ALLIAGE 17 X 7,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,224.57,299.43,NaN
54,000AK18MP01,ALLOY WHEEL 18 x 7.5,NaN,JANTES EN ALLIAGE 18 X 7.5,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,261.69,348.92,NaN
56,000AK18MP03,ALLOY WHEEL 18 x 7.5,NaN,JANTES EN ALLIAGE 18 X 7.5,NaN,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,261.75,349.00,NaN
66,AMH40AP016B,STEEL WHEEL 16 x 6.5,Boxed single,"JANTES EN ACIER - 16"" x 6.5",Unité individuelle en boîte,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,68.00,94.99,NaN
67,AMH40AP016BP,STEEL WHEEL 16 x 6.5,Boxed pallet. 30 units per pallet,"JANTES EN ACIER - 16"" x 6.5",Roue dans une boîte sur une palette. 30 roues ...,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,62.00,94.99,NaN
68,AMH40AP017B,STEEL WHEEL 17 x 7,Boxed single,"JANTES EN ACIER - 17"" x 7",Unité individuelle en boîte,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,75.00,103.99,NaN
69,AMH40AP017BP,STEEL WHEEL 17 x 7,Boxed pallet. 24 units per pallet,"JANTES EN ACIER - 17"" x 7",Roue dans une boîte sur une palette. 24 roues ...,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,70.00,103.99,NaN
70,AMH40AP018B,STEEL WHEEL 18 x 7.5,Boxed single,JANTES EN ACIER - 18'' x 7.5,Unité individuelle en boîte,2021,2024,K5,LX,EX,GT-Line,GT,NaN,NaN,NaN,84.00,115.99,NaN


In [98]:
'''

20 records missing labour hours

3 part number have the value in other records, so they can be filled. 

Assumption: part with the same part numbers would be the same and would take the same labour hours. 

The remaining 17 part# (about 281 records) need check with the Manufacture to know how they should be filled.

'''

'\n\n20 records missing labour hours\n\n3 part number have the value in other records, so they can be filled. \n\nAssumption: part with the same part numbers would be the same and would take the same labour hours. \n\nThe remaining 17 part# (about 281 records) need check with the Manufacture to know how they should be filled.\n\n'

In [99]:
'''

Checking records missing applicability 

'''

failed_rows[
    (failed_rows["dq_rule"]=="any_not_null")
    & (failed_rows["dq_columns"].str.contains("trim_"))
    ]["part_number"].value_counts()
 


part_number
DWH46AP000     4
TBD            2
000AM481004    1
9999611001     1
Name: count, dtype: int64

In [115]:
def fix_swapped_years(df, min_col='model_year_from', max_col='model_year_to', suffix='_fixed'):
    """Swap min/max years on rows where min_col > max_col (a typo), without touching the originals.

    Adds:
      source_row        the row's index in df, so every record traces back to where it came from
      <min_col>_fixed   from-year, swapped where needed
      <max_col>_fixed   to-year, swapped where needed
      years_swapped     True on rows that were swapped
      year_fix_note     what was done on that row ('' if nothing)
    """
    missing = [c for c in (min_col, max_col) if c not in df.columns]
    if missing:
        raise KeyError(f"Column(s) not in df: {missing}")

    df = df.loc[:, ~df.columns.duplicated()].copy()   # copy: the input df is left unchanged
    new_min, new_max = f"{min_col}{suffix}", f"{max_col}{suffix}"
    clash = [c for c in (new_min, new_max, 'years_swapped', 'year_fix_note') if c in df.columns]
    if clash:
        raise ValueError(f"df already has {clash}; run on the original df or pass another suffix")

    if 'source_row' not in df.columns:
        df.insert(0, 'source_row', df.index)

    lo = pd.to_numeric(df[min_col], errors='coerce')
    hi = pd.to_numeric(df[max_col], errors='coerce')
    swap = (lo > hi).fillna(False)          # only rows where both are numbers and from > to

    df[new_min] = lo.where(~swap, hi)
    df[new_max] = hi.where(~swap, lo)
    df['years_swapped'] = swap
    df['year_fix_note'] = ''
    df.loc[swap, 'year_fix_note'] = (
        f"{min_col}/{max_col} swapped: " + lo[swap].astype('Int64').astype(str)
        + "-" + hi[swap].astype('Int64').astype(str) + " -> "
        + hi[swap].astype('Int64').astype(str) + "-" + lo[swap].astype('Int64').astype(str)
    )
    # Blanks or text the swap couldn't judge: left as is, noted for review
    unreadable = lo.isna() | hi.isna()
    df.loc[unreadable, 'year_fix_note'] = 'year missing or not a number: not checked'

    # Whole-number years stay whole numbers (blanks allowed)
    df[new_min] = df[new_min].round().astype('Int64')
    df[new_max] = df[new_max].round().astype('Int64')

    print(f"Swapped years on {int(swap.sum())} rows"
          + (f"; {int(unreadable.sum())} rows with missing/non-numeric years left for review"
             if unreadable.any() else ""))
    return df

In [144]:


def expand_trims(df, trim_keyword='trim_', trim_cols=None, output_col='Trim',
                 key_cols=('part_number', 'model', 'Year'), keep_original_trims=False):
    """Copy each row once per trim it applies to, tagging transformations and issues (nothing is dropped).

    trim_keyword: trim columns are the columns whose name contains it (trim_1 ... trim_7).
    trim_cols:    pass the list yourself to override the keyword search.
    output_col:   new column holding the single trim each row applies to.
    key_cols:     columns that identify a part on a vehicle (used to spot the same fitment in several rows).

    Added columns:
      source_row         the row's index in the original df (kept if already there)
      <output_col>       the trim this row applies to (blank on orphaned records)
      trim_source_col    the trim column it came from
      trim_raw_value     the value exactly as it was in the cell
      transformations    what was done:  expanded_by_trim; spaces_cleaned; kept_as_single_row_no_trim
      issues             what to check:  orphaned_no_trim; duplicate_trim_in_row;
                                         possible_multiple_trims_in_cell; same_part_vehicle_trim_in_other_row
      has_issue          True if this row has any issue
      record_has_issue   True on every row of a source record that has an issue anywhere
    """
    # --- input checks ---
    df = df.loc[:, ~df.columns.duplicated()].copy()
    new_cols = [output_col, 'trim_source_col', 'trim_raw_value', 'transformations',
                'issues', 'has_issue', 'record_has_issue']
    if trim_cols is None:
        trim_cols = [c for c in df.columns
                     if trim_keyword.casefold() in str(c).casefold() and c not in new_cols]
    if not trim_cols:
        raise KeyError(f"No columns containing '{trim_keyword}' found; pass trim_cols=[...]")
    missing = [c for c in trim_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Trim column(s) not in df: {missing}")
    clash = [c for c in new_cols if c in df.columns]
    if clash:
        raise ValueError(f"df already has {clash}; run on the df before trim expansion")
    key_cols = [c for c in key_cols if c in df.columns]
    print(f"Trim columns used: {trim_cols}")

    if 'source_row' not in df.columns:
        df.insert(0, 'source_row', df.index)
    df['_pos'] = range(len(df))

    # --- one row per (record, trim cell) ---
    id_cols = [c for c in df.columns if c not in trim_cols]
    long = df.melt(id_vars=id_cols, value_vars=trim_cols,
                   var_name='trim_source_col', value_name='trim_raw_value')
    clean = long['trim_raw_value'].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    blank = clean.isna() | clean.eq('') | clean.str.casefold().isin(['nan', 'none', 'null'])
    long[output_col] = clean
    kept = long[~blank]          # empty trim cells only: no vehicle to copy the row for

    # --- records with no trim at all stay in, as one row with a blank trim ---
    orphans = df.loc[~df['_pos'].isin(kept['_pos']), id_cols].assign(
        trim_source_col=pd.NA, trim_raw_value=pd.NA, **{output_col: pd.NA})

    out = pd.concat([kept, orphans], ignore_index=True)
    out['_col_order'] = out['trim_source_col'].map({c: i for i, c in enumerate(trim_cols)}).fillna(-1)
    out = out.sort_values(['_pos', '_col_order']).reset_index(drop=True)

    # --- flags ---
    orphan = out[output_col].isna()
    out['_key'] = out[output_col].str.casefold()
    spaces = ~orphan & (out['trim_raw_value'].astype(str) != out[output_col].astype(str))
    multi = ~orphan & out[output_col].str.contains(r'[,;/+]|\band\b', case=False, regex=True, na=False)
    dup_row = ~orphan & out.duplicated(['_pos', '_key'])
    cross = pd.Series(False, index=out.index)
    if key_cols:
        n_sources = out[~orphan].groupby([*key_cols, '_key'])['source_row'].transform('nunique')
        cross[~orphan] = n_sources.gt(1)

    def join_tags(flags):
        tags = pd.Series('', index=out.index)
        for name, mask in flags.items():
            tags = tags.mask(mask, tags.where(tags.eq(''), tags + '; ') + name)
        return tags

    out['transformations'] = join_tags({'expanded_by_trim': ~orphan,
                                        'spaces_cleaned': spaces,
                                        'kept_as_single_row_no_trim': orphan})
    out['issues'] = join_tags({'orphaned_no_trim': orphan,
                               'duplicate_trim_in_row': dup_row,
                               'possible_multiple_trims_in_cell': multi,
                               'same_part_vehicle_trim_in_other_row': cross})
    out['has_issue'] = out['issues'].ne('')
    out['record_has_issue'] = out.groupby('source_row')['has_issue'].transform('any')

    # --- finish: review columns first ---
    out = out.drop(columns=['_pos', '_col_order', '_key'])
    if keep_original_trims:
        out = out.merge(df[['source_row', *trim_cols]], on='source_row', how='left')
    front = ['source_row', *key_cols, output_col, 'trim_source_col', 'trim_raw_value',
             'has_issue', 'record_has_issue', 'issues', 'transformations']
    out = out[front + [c for c in out.columns if c not in front]]

    # --- report ---
    def tag_counts(col):
        return out[col].str.split('; ').explode().loc[lambda s: s.ne('')].value_counts()

    print("Trim expansion report")
    print("-" * 60)
    print(f"Records in:                 {len(df)}")
    print(f"Rows out:                   {len(out)}")
    print(f"Unique trim names:          {out[output_col].str.casefold().nunique()}")
    print(f"Records with issues:        {out.loc[out['has_issue'], 'source_row'].nunique()}")
    print(f"Rows with issues:           {int(out['has_issue'].sum())}")
    print("\nTransformations (rows):")
    print(tag_counts('transformations').to_string())
    issues = tag_counts('issues')
    print("\nIssues (rows):")
    print(issues.to_string() if len(issues) else "none")
    return out

In [112]:


def expand_years(df, min_col='model_year_from', max_col='model_year_to', year_col='Year',
                 year_bounds=(1900, 2100), max_span=30):
    """One row per year in each record's [min_col, max_col] range, both ends included.

    Returns (expanded, invalid):
      expanded: the valid rows, repeated once per year, with a new year_col column and a
                source_row column (the row's index in df) so each copy traces back to its record
      invalid:  rows that could not be expanded, with an 'expand_issue' column saying why
    """
    # --- input checks: fail early with a clear message ---
    missing = [c for c in (min_col, max_col) if c not in df.columns]
    if missing:
        raise KeyError(f"Column(s) not in df: {missing}")
    if year_col in df.columns:
        raise ValueError(f"df already has a '{year_col}' column; pass another year_col name")

    df = df.loc[:, ~df.columns.duplicated()]   # a duplicated column would make df[col] a table

    lo = pd.to_numeric(df[min_col], errors='coerce')   # text like "2021" -> 2021, junk -> NaN
    hi = pd.to_numeric(df[max_col], errors='coerce')

    # --- find rows that can't be expanded, first problem wins ---
    issue = pd.Series('', index=df.index, dtype=object)
    def flag(mask, text):
        issue[mask & issue.eq('')] = text

    flag(lo.isna() | hi.isna(), 'year missing or not a number')
    flag((lo % 1 != 0) | (hi % 1 != 0), 'year is not a whole number')
    flag((lo < year_bounds[0]) | (hi > year_bounds[1]),
         f'year outside {year_bounds[0]}-{year_bounds[1]}')
    flag(lo > hi, f'{min_col} is after {max_col}')
    flag(hi - lo + 1 > max_span, f'range longer than {max_span} years')

    valid = issue.eq('')
    invalid = df[~valid].assign(expand_issue=issue[~valid])
    good = df[valid]

    # --- repeat each valid row once per year ---
    starts = lo[valid].astype(int).to_numpy()
    counts = (hi[valid].astype(int) - lo[valid].astype(int) + 1).to_numpy()
    pos = np.repeat(np.arange(len(good)), counts)                     # which row each copy comes from
    offset = np.arange(counts.sum()) - np.repeat(counts.cumsum() - counts, counts)  # 0, 1, 2... per row

    expanded = good.iloc[pos].copy()
    if 'source_row' not in expanded.columns:
        expanded.insert(0, 'source_row', good.index.to_numpy()[pos])
    expanded[year_col] = starts[pos] + offset
    expanded = expanded.reset_index(drop=True)

    print(f"Expanded {len(good)} rows into {len(expanded)} year rows"
          + (f"; {len(invalid)} rows not expanded (see invalid['expand_issue'])" if len(invalid) else ""))
    return expanded, invalid


access_clean, year_issues = expand_years(raw_cleaned_df)                 # uses model_year_from / model_year_to
# yearly_df, year_issues = expand_years(df, 'min_year', 'max_year')   # other column names



Expanded 2461 rows into 6963 year rows; 1 rows not expanded (see invalid['expand_issue'])


In [114]:
year_issues

,part_number,description_en,comments_en,description_fr,comments_fr,model_year_from,model_year_to,model,trim_1,trim_2,trim_3,trim_4,trim_5,trim_6,trim_7,dealer_cost,MSRP,Labour_hours,expand_issue
366,000KCPENCR5,T/U PEN CR5,NaN,T/U PEN CR5,NaN,2025,2023,Sorento HEV PE,LX,EX,SX,NaN,NaN,NaN,NaN,11.5,19.18,0.0,model_year_from is after model_year_to


In [ ]:
# correcting the data, swapped year_from/year_to to conflicting records 

years_fixed_df = fix_swapped_years(raw_cleaned_df)



Swapped years on 1 rows


In [ ]:
# What was changed: the originals and the fixed values side by side
years_fixed_df.loc[years_fixed_df['years_swapped'],
                   ['source_row', 'part_number', 'model',
                    'model_year_from', 'model_year_to',
                    'model_year_from_fixed', 'model_year_to_fixed', 'year_fix_note']]

# Expand years from the fixed columns
yearly_df, year_issues = expand_years(years_fixed_df, 'model_year_from_fixed', 'model_year_to_fixed')

Expanded 2462 rows into 6966 year rows


In [ ]:

years_fixed_df.loc[years_fixed_df['years_swapped'],
                   ['source_row', 'part_number', 'model',
                    'model_year_from', 'model_year_to',
                    'model_year_from_fixed', 'model_year_to_fixed', 'year_fix_note']]


,source_row,part_number,model,model_year_from,model_year_to,model_year_from_fixed,model_year_to_fixed,year_fix_note
366,366,000KCPENCR5,Sorento HEV PE,2025,2023,2023,2025,model_year_from/model_year_to swapped: 2025-20...


In [145]:
vehicle_parts_df = expand_trims(yearly_df)

         # clean rows, ready to use

Trim columns used: ['trim_1', 'trim_2', 'trim_3', 'trim_4', 'trim_5', 'trim_6', 'trim_7']
Trim expansion report
------------------------------------------------------------
Records in:                 6966
Rows out:                   27576
Unique trim names:          60
Records with issues:        802
Rows with issues:           3495

Transformations (rows):
transformations
expanded_by_trim              27559
kept_as_single_row_no_trim       17

Issues (rows):
issues
possible_multiple_trims_in_cell        3437
same_part_vehicle_trim_in_other_row      34
orphaned_no_trim                         17
duplicate_trim_in_row                     9


In [ ]:
# vehicle_parts_df[vehicle_parts_df['has_issue']]                  # only the rows with a problem
# vehicle_parts_df[vehicle_parts_df['record_has_issue']]           # every row of any record with a problem
vehicle_parts_df[vehicle_parts_df['issues'].str.contains('orphaned_no_trim')]   # one issue type
# vehicle_parts_df[~vehicle_parts_df['has_issue']]        

,source_row,part_number,model,Year,Trim,trim_source_col,trim_raw_value,has_issue,record_has_issue,issues,...,comments_fr,model_year_from,model_year_to,dealer_cost,MSRP,Labour_hours,model_year_from_fixed,model_year_to_fixed,years_swapped,year_fix_note
10563,1029,DWH46AP000,Sportage HEV PE,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2026,2026,186.29,266.13,0.5,2026,2026,False,
10762,1104,DWH46AP000,Sportage HEV,2023,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
10763,1104,DWH46AP000,Sportage HEV,2024,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
10764,1104,DWH46AP000,Sportage HEV,2025,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
11155,1179,DWH46AP000,Sportage PHEV PE,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2026,2026,186.29,266.13,0.5,2026,2026,False,
11358,1257,DWH46AP000,Sportage PHEV,2023,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
11359,1257,DWH46AP000,Sportage PHEV,2024,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
11360,1257,DWH46AP000,Sportage PHEV,2025,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
25566,2271,TBD,K4 Hatchback,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,En développement,2026,2026,99.99,139.99,0.0,2026,2026,False,
25614,2291,TBD,K4 Hatchback,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,En développement,2026,2026,268.01,382.87,0.5,2026,2026,False,


In [ ]:
'''

Parts missing trim applicability

Need to define these data points - upcoming/ other 

Determine what to do with them based on definitions. This feature should be applied to other OEMs too.
 
'''

vehicle_parts_df[vehicle_parts_df['issues'].str.contains('orphaned_no_trim')]   # one issue type
 

,source_row,part_number,model,Year,Trim,trim_source_col,trim_raw_value,has_issue,record_has_issue,issues,...,comments_fr,model_year_from,model_year_to,dealer_cost,MSRP,Labour_hours,model_year_from_fixed,model_year_to_fixed,years_swapped,year_fix_note
10563,1029,DWH46AP000,Sportage HEV PE,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2026,2026,186.29,266.13,0.5,2026,2026,False,
10762,1104,DWH46AP000,Sportage HEV,2023,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
10763,1104,DWH46AP000,Sportage HEV,2024,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
10764,1104,DWH46AP000,Sportage HEV,2025,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
11155,1179,DWH46AP000,Sportage PHEV PE,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2026,2026,186.29,266.13,0.5,2026,2026,False,
11358,1257,DWH46AP000,Sportage PHEV,2023,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
11359,1257,DWH46AP000,Sportage PHEV,2024,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
11360,1257,DWH46AP000,Sportage PHEV,2025,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,NaN,2023,2025,186.29,266.13,0.5,2023,2025,False,
25566,2271,TBD,K4 Hatchback,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,En développement,2026,2026,99.99,139.99,0.0,2026,2026,False,
25614,2291,TBD,K4 Hatchback,2026,<NA>,<NA>,<NA>,True,True,orphaned_no_trim,...,En développement,2026,2026,268.01,382.87,0.5,2026,2026,False,


In [160]:
'''
Unique model names:

- they are all valid, they contain model name and power train or transmission 

'''

vehicle_parts_df[vehicle_parts_df['issues'].str.contains('possible_multiple_trims_in_cell')]["trim_raw_value"].unique()
# [["Year", "model", "trim_raw_value"]]   # one issue type

array(['LX+', 'EX+', 'SX+', 'Premium +', 'Wind +', 'LX+ MT', 'LX+ IVT'],
      dtype=object)

In [165]:
'''

After expanding by trim levels, some records seem to have been duplicated (same rows )

We'll need to validate this by 

'''

vehicle_parts_df[vehicle_parts_df['issues'].str.contains('same_part_vehicle_trim_in_other_row')].columns

Index(['source_row', 'part_number', 'model', 'Year', 'Trim', 'trim_source_col',
       'trim_raw_value', 'has_issue', 'record_has_issue', 'issues',
       'transformations', 'description_en', 'comments_en', 'description_fr',
       'comments_fr', 'model_year_from', 'model_year_to', 'dealer_cost',
       'MSRP', 'Labour_hours', 'model_year_from_fixed', 'model_year_to_fixed',
       'years_swapped', 'year_fix_note'],
      dtype='str')

In [188]:
dupes = vehicle_parts_df[vehicle_parts_df['issues'].str.contains('same_part_vehicle_trim_in_other_row')]
dupes.sort_values(['part_number', 'model', 'Year', 'Trim'])

# Which columns differ between the duplicate records (empty = exact duplicates, safe to drop)
key = ['part_number', 'model', 'Year', 'Trim','MSRP', 'Labour_hours', 'description_en']
dupes.groupby(key).nunique().loc[:, lambda d: (d > 1).any()]

Empty DataFrame
Columns: []
Index: [(DOH14AP100, EV9, 2024, Land AWD, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Land AWD, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Land Plus Package, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Land Plus Package, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Light RWD (Short Range), 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Light RWD (Short Range), 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Wind RWD, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2024, Wind RWD, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Land AWD, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Land AWD, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Land Plus Package, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Land Plus Package, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Light RWD (Short Range), 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Light RWD (Short Range), 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Wind RWD, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2025, Wind RWD, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Land AWD, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Land AWD, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Land Plus Package, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Land Plus Package, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Light RWD (Short Range), 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Light RWD (Short Range), 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Wind RWD, 279.81, 0.0, CARPET MATS (7P)), (DOH14AP100, EV9, 2026, Wind RWD, 279.81, 0.0, COMPLETE ASSY-CARPET MATS (7P)), (TBD, K4 Hatchback, 2026, EX, 205.61, 1.0, BLOCK HEATER - 2.0L), (TBD, K4 Hatchback, 2026, EX, 238.49, 0.5, WINDOW VISOR), (TBD, K4 Hatchback, 2026, EX, 969.8, 0.3, CROSS BARS), (TBD, K4 Hatchback, 2026, EX Prem, 205.61, 1.0, BLOCK HEATER - 2.0L), (TBD, K4 Hatchback, 2026, EX Prem, 238.49, 0.5, WINDOW VISOR), (TBD, K4 Hatchback, 2026, EX+, 205.61, 1.0, BLOCK HEATER - 2.0L), (TBD, K4 Hatchback, 2026, EX+, 238.49, 0.5, WINDOW VISOR), (TBD, K4 Hatchback, 2026, LX, 205.61, 1.0, BLOCK HEATER - 2.0L), (TBD, K4 Hatchback, 2026, LX, 238.49, 0.5, WINDOW VISOR), (TBD, K4 Hatchback, 2026, LX, 969.8, 0.3, CROSS BARS)]

In [ ]:


key = ['part_number', 'model', 'Year', 'Trim', 'MSRP', 'Labour_hours']
desc_cols = ['description_en', 'description_fr']

df = vehicle_parts_df

# Groups (same key) where the descriptions differ
differs = df.groupby(key, dropna=False)[desc_cols].transform('nunique', dropna=False).gt(1).any(axis=1)
desc_conflicts = df[differs].sort_values(key + ['source_row'])



In [190]:

same_except_desc_df = desc_conflicts[['source_row', *key, *desc_cols]]


In [ ]:
'''
same_part_vehicle_trim_in_other_row


Outcome: 
- Confirmed the records have same data except description. 


To do:
- Might need to add this as a data quality item
- Since the goal is to make sure that we are not duplicating, we will make this pass as long as the price, labour hours, and description are different. With this, th records will be unique all through. make it a layers sequence o checks.
- Come up with conditions that help us to make sure that it is not an issue.

'''

same_except_desc_df["description_en"].value_counts()
 

description_en
COMPLETE ASSY-CARPET MATS (7P)    12
CARPET MATS (7P)                  12
Name: count, dtype: int64

In [ ]:

same_except_desc_df["part_number"].value_counts()
 

part_number
DOH14AP100    24
Name: count, dtype: int64

In [ ]:

'''
Trim duplicated in the row

Thought process:

- Is this and error for duplicating or did they mean to write a diff trim.

Logic todo assuming it's a simple duplication error - we'll go with this one for now:
- De-duplicate the trim options in a row, but make sure that they are identical.

'''

vehicle_parts_df[vehicle_parts_df['issues'].str.contains('duplicate_trim_in_row')]

,source_row,part_number,model,Year,Trim,trim_source_col,trim_raw_value,has_issue,record_has_issue,issues,...,comments_fr,model_year_from,model_year_to,dealer_cost,MSRP,Labour_hours,model_year_from_fixed,model_year_to_fixed,years_swapped,year_fix_note
8742,870,DWF21ACA00CA,Sportage PE,2026,X-Line Limited,trim_6,X-Line Limited,True,True,duplicate_trim_in_row,...,NaN,2026,2026,617.29,1028.82,0.3,2026,2026,False,
9351,947,DWF21ACA00CA,Sportage,2023,X-Line Limited,trim_6,X-Line Limited,True,True,duplicate_trim_in_row,...,NaN,2023,2025,617.29,1028.82,0.3,2023,2025,False,
9354,947,DWF21ACA00CA,Sportage,2024,X-Line Limited,trim_6,X-Line Limited,True,True,duplicate_trim_in_row,...,NaN,2023,2025,617.29,1028.82,0.3,2023,2025,False,
9357,947,DWF21ACA00CA,Sportage,2025,X-Line Limited,trim_6,X-Line Limited,True,True,duplicate_trim_in_row,...,NaN,2023,2025,617.29,1028.82,0.3,2023,2025,False,
26758,2424,H9H22AP100,Rio/Rio5,2018,LX,trim_5,LX,True,True,duplicate_trim_in_row,...,NaN,2018,2022,6.76,7.95,0.5,2018,2022,False,
26761,2424,H9H22AP100,Rio/Rio5,2019,LX,trim_5,LX,True,True,duplicate_trim_in_row,...,NaN,2018,2022,6.76,7.95,0.5,2018,2022,False,
26764,2424,H9H22AP100,Rio/Rio5,2020,LX,trim_5,LX,True,True,duplicate_trim_in_row,...,NaN,2018,2022,6.76,7.95,0.5,2018,2022,False,
26767,2424,H9H22AP100,Rio/Rio5,2021,LX,trim_5,LX,True,True,duplicate_trim_in_row,...,NaN,2018,2022,6.76,7.95,0.5,2018,2022,False,
26770,2424,H9H22AP100,Rio/Rio5,2022,LX,trim_5,LX,True,True,duplicate_trim_in_row,...,NaN,2018,2022,6.76,7.95,0.5,2018,2022,False,


In [ ]:


vehicle_parts_df['issues'].value_counts()

issues
                                                                        24081
possible_multiple_trims_in_cell                                          3435
same_part_vehicle_trim_in_other_row                                        32
orphaned_no_trim                                                           17
duplicate_trim_in_row                                                       9
possible_multiple_trims_in_cell; same_part_vehicle_trim_in_other_row        2
Name: count, dtype: int64

In [ ]:

vehicle_parts_df.head()


,source_row,part_number,description_en,comments_en,description_fr,comments_fr,model_year_from,model_year_to,model,dealer_cost,MSRP,Labour_hours,model_year_from_fixed,model_year_to_fixed,years_swapped,year_fix_note,Year,trim_source_col,Trim
0,0,L3F03AP000,ACCESSORY HITCH,NOT COMPATIBLE WITH GT 2.5T TRIM,ACCESSOIRES DE TOIT,NE PAS COMPATIBLE AVEC GT 2.5T,2021,2024,K5,22.92,26.96,0.5,2021,2024,False,,2021,trim_1,LX
1,0,L3F03AP000,ACCESSORY HITCH,NOT COMPATIBLE WITH GT 2.5T TRIM,ACCESSOIRES DE TOIT,NE PAS COMPATIBLE AVEC GT 2.5T,2021,2024,K5,22.92,26.96,0.5,2021,2024,False,,2021,trim_2,EX
2,0,L3F03AP000,ACCESSORY HITCH,NOT COMPATIBLE WITH GT 2.5T TRIM,ACCESSOIRES DE TOIT,NE PAS COMPATIBLE AVEC GT 2.5T,2021,2024,K5,22.92,26.96,0.5,2021,2024,False,,2021,trim_3,GT-Line
3,0,L3F03AP000,ACCESSORY HITCH,NOT COMPATIBLE WITH GT 2.5T TRIM,ACCESSOIRES DE TOIT,NE PAS COMPATIBLE AVEC GT 2.5T,2021,2024,K5,22.92,26.96,0.5,2021,2024,False,,2022,trim_1,LX
4,0,L3F03AP000,ACCESSORY HITCH,NOT COMPATIBLE WITH GT 2.5T TRIM,ACCESSOIRES DE TOIT,NE PAS COMPATIBLE AVEC GT 2.5T,2021,2024,K5,22.92,26.96,0.5,2021,2024,False,,2022,trim_2,EX


In [126]:
# yearly_df.columns

access_clean = yearly_df.copy()

In [132]:
import re

def match_model_years(model_catalogue, accessories_list,
                      catalogue_cols=('ModelName', 'ModelYear'),
                      accessory_cols=('model', 'Year'),
                      partial=False):
    """Compare (model name, year) pairs between the model catalogue and the accessories list.

    catalogue_cols / accessory_cols: (model name column, year column) in each df.
    partial=False: names must be equal (ignoring case, extra spaces, spacing around / - &).
    partial=True:  the accessory name is searched inside the catalogue names for the same year,
                   as whole words ("sorento" finds "sorento hev", not "sorentox").

    Returns audit: one row per unique accessory (model, year) pair, plus catalogue pairs no accessory matched:
      status            matched | accessory_only | catalogue_only
      match_type        exact | partial ('' if not matched)
      model_key, year   normalized accessory name (catalogue name for catalogue_only rows) and year
      accessory_names   original accessory spellings
      catalogue_names   original spellings of every catalogue name matched
      accessory_rows, catalogue_rows   how many rows each side has behind the pair
      reason            for accessory_only: name missing from catalogue, or only the year missing
    """
    def normalize(s):
        return (s.astype('string').str.strip().str.casefold()
                 .str.replace(r'\s*([/\-&])\s*', r'\1', regex=True)   # "rio / rio5" -> "rio/rio5"
                 .str.replace(r'\s+', ' ', regex=True))

    def prepare(df, cols, side):
        name_col, year_col = cols
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise KeyError(f"{side}: column(s) not in df: {missing}")
        df = df.loc[:, ~df.columns.duplicated()]
        out = pd.DataFrame({'name': df[name_col].astype('string').str.strip(),
                            'year': pd.to_numeric(df[year_col], errors='coerce')})
        out['model_key'] = normalize(df[name_col])
        bad = out['model_key'].isna() | out['model_key'].eq('') | out['year'].isna() | (out['year'] % 1 != 0)
        if bad.any():
            print(f"  {side}: {int(bad.sum())} of {len(out)} rows skipped (blank model name or year not a whole number)")
        out = out[~bad].copy()
        out['year'] = out['year'].astype(int)
        return (out.groupby(['model_key', 'year'])
                   .agg(names=('name', lambda s: ', '.join(sorted(s.unique()))), rows=('name', 'size'))
                   .reset_index())

    def found_in(acc_key, cat_key):
        if acc_key == cat_key:
            return 'exact'
        if partial and re.search(rf'(?<![a-z0-9]){re.escape(acc_key)}(?![a-z0-9])', cat_key):
            return 'partial'
        return None

    print(f"Model / year match report ({'partial' if partial else 'exact'} name search)")
    print("-" * 60)
    cat = prepare(model_catalogue, catalogue_cols, 'catalogue')
    acc = prepare(accessories_list, accessory_cols, 'accessories')

    cat_by_year = {y: g for y, g in cat.groupby('year')}
    all_cat_keys = cat.groupby('model_key')['year'].agg(lambda y: sorted(y.unique()))
    used_cat = set()   # (catalogue key, year) pairs matched by some accessory pair

    rows = []
    for a in acc.itertuples(index=False):
        candidates = cat_by_year.get(a.year, cat.iloc[0:0])
        hits = [(c, found_in(a.model_key, c.model_key)) for c in candidates.itertuples(index=False)]
        hits = [(c, t) for c, t in hits if t]

        if hits:
            used_cat.update((c.model_key, c.year) for c, _ in hits)
            rows.append({'status': 'matched',
                         'match_type': 'exact' if any(t == 'exact' for _, t in hits) else 'partial',
                         'model_key': a.model_key, 'year': a.year,
                         'accessory_names': a.names,
                         'catalogue_names': ' | '.join(sorted({c.names for c, _ in hits})),
                         'accessory_rows': a.rows, 'catalogue_rows': sum(c.rows for c, _ in hits),
                         'reason': ''})
        else:
            # Same name test against every year: is it the name or only the year that's missing?
            years = sorted({y for k, ys in all_cat_keys.items() if found_in(a.model_key, k) for y in ys})
            reason = (f"year not in catalogue (catalogue has {years[0]}-{years[-1]})" if years
                      else 'model name not in catalogue')
            rows.append({'status': 'accessory_only', 'match_type': '',
                         'model_key': a.model_key, 'year': a.year,
                         'accessory_names': a.names, 'catalogue_names': '',
                         'accessory_rows': a.rows, 'catalogue_rows': 0, 'reason': reason})

    for c in cat.itertuples(index=False):
        if (c.model_key, c.year) not in used_cat:
            rows.append({'status': 'catalogue_only', 'match_type': '',
                         'model_key': c.model_key, 'year': c.year,
                         'accessory_names': '', 'catalogue_names': c.names,
                         'accessory_rows': 0, 'catalogue_rows': c.rows, 'reason': ''})

    audit = (pd.DataFrame(rows, columns=['status', 'match_type', 'model_key', 'year', 'accessory_names',
                                         'catalogue_names', 'accessory_rows', 'catalogue_rows', 'reason'])
               .sort_values(['status', 'model_key', 'year']).reset_index(drop=True))

    # --- report ---
    acc_audit = audit[audit['status'] != 'catalogue_only']
    n_acc, n_match = len(acc_audit), (acc_audit['status'] == 'matched').sum()
    print(f"Accessory model/year pairs:  {n_acc}")
    if n_acc:
        print(f"  matched in catalogue:      {n_match} ({n_match / n_acc:.0%})")
        if partial:
            print(f"    exact:                   {(acc_audit['match_type'] == 'exact').sum()}")
            print(f"    partial only:            {(acc_audit['match_type'] == 'partial').sum()}")
        print(f"  not in catalogue:          {n_acc - n_match}")
    print(f"Catalogue pairs with no accessories: {(audit['status'] == 'catalogue_only').sum()}")
    unmatched = acc_audit[acc_audit['status'] == 'accessory_only']
    if not unmatched.empty:
        print("\nUnmatched accessory pairs by reason:")
        print(unmatched['reason'].str.replace(r' \(.*', '', regex=True).value_counts().to_string())
        print(f"Accessory rows affected: {unmatched['accessory_rows'].sum()}")
    return audit

In [100]:
'''
Reviewing the vehicle catalogue dataset.
'''

veh_catalogue_df = pd.read_csv("datasets/kia_models_catalogue.csv")



In [101]:
veh_catalogue_df.head()

,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,Style_ID,TrimName,engine_type
0,Cadenza Premium Sedan,NaN,KIA,NaN,F6S4J5A1JGG005,2019,NaN,NaN,404251,NaN,NaN
1,Cadenza Limited Sedan,NaN,KIA,NaN,F6S4J5A1JGGAJH,2019,NaN,NaN,404252,NaN,NaN
2,Forte LX Manual,NaN,KIA,NaN,DLS42G617DD075,2019,NaN,NaN,403995,NaN,NaN
3,Forte LX IVT,NaN,KIA,NaN,DLS42G61VDD076,2019,NaN,NaN,403996,NaN,NaN
4,Forte EX IVT,NaN,KIA,NaN,DLS42G61VHH129,2019,NaN,NaN,403997,NaN,NaN


In [102]:

veh_catalogue_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1736 entries, 0 to 1735
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Description   1736 non-null   str    
 1   Drivetrain    0 non-null      float64
 2   Manufacturer  1736 non-null   str    
 3   ModelName     0 non-null      float64
 4   ModelNumber   1736 non-null   str    
 5   ModelYear     1736 non-null   int64  
 6   Package       0 non-null      float64
 7   PassDoors     0 non-null      float64
 8   Style_ID      986 non-null    str    
 9   TrimName      0 non-null      float64
 10  engine_type   0 non-null      float64
dtypes: float64(6), int64(1), str(4)
memory usage: 149.3 KB


In [103]:
veh_catalogue_df = veh_catalogue_df.dropna(axis=1)

In [104]:

veh_catalogue_df


,Description,Manufacturer,ModelNumber,ModelYear
0,Cadenza Premium Sedan,KIA,F6S4J5A1JGG005,2019
1,Cadenza Limited Sedan,KIA,F6S4J5A1JGGAJH,2019
2,Forte LX Manual,KIA,DLS42G617DD075,2019
3,Forte LX IVT,KIA,DLS42G61VDD076,2019
4,Forte EX IVT,KIA,DLS42G61VHH129,2019
...,...,...,...,...
1731,EV9 GT AWD,KIA,DOW6YDZ7ZBS0BB0QQ,2027
1732,EV9 Land w/GT-Line 6P Pkg AWD,KIA,DOW6YBZ7ZBS0HH859,2027
1733,Sortento PHEV LX AWD,KIA,HCW6D5G1GEV2HH01C,2027
1734,Sortento PHEV EX AWD,KIA,HCW6D5G1GEV2KK149,2027


In [ ]:
remove_duplicates(veh_catalogue_df)

Removed 870 duplicate rows (1736 -> 866)


,Description,Manufacturer,ModelNumber,ModelYear
0,Cadenza Premium Sedan,KIA,F6S4J5A1JGG005,2019
1,Cadenza Limited Sedan,KIA,F6S4J5A1JGGAJH,2019
2,Forte LX Manual,KIA,DLS42G617DD075,2019
3,Forte LX IVT,KIA,DLS42G61VDD076,2019
4,Forte EX IVT,KIA,DLS42G61VHH129,2019
...,...,...,...,...
861,EV9 GT AWD,KIA,DOW6YDZ7ZBS0BB0QQ,2027
862,EV9 Land w/GT-Line 6P Pkg AWD,KIA,DOW6YBZ7ZBS0HH859,2027
863,Sortento PHEV LX AWD,KIA,HCW6D5G1GEV2HH01C,2027
864,Sortento PHEV EX AWD,KIA,HCW6D5G1GEV2KK149,2027


In [135]:
audit = match_model_years(veh_catalogue_df, yearly_df,
                          catalogue_cols=('Description', 'ModelYear'),
                          accessory_cols=('model', 'Year'),
                          partial=True)


Model / year match report (partial name search)
------------------------------------------------------------
Accessory model/year pairs:  110
  matched in catalogue:      68 (62%)
    exact:                   0
    partial only:            68
  not in catalogue:          42
Catalogue pairs with no accessories: 395

Unmatched accessory pairs by reason:
reason
model name not in catalogue    35
year not in catalogue           7
Accessory rows affected: 2378


In [136]:
audit[audit['match_type'] == 'partial'][['model_key', 'year', 'accessory_names', 'catalogue_names']]  # review these

,model_key,year,accessory_names,catalogue_names
437,carnival,2022,Carnival,Carnival EX FWD | Carnival EX FWD *Ltd Avail* ...
438,carnival,2023,Carnival,Carnival EX | Carnival EX+ | Carnival LX | Car...
439,carnival,2024,Carnival,Carnival EX | Carnival EX+ | Carnival LX | Car...
440,carnival hev,2025,Carnival HEV,Carnival HEV EX | Carnival HEV LX+ | Carnival ...
441,carnival hev,2026,Carnival HEV,Carnival HEV EX | Carnival HEV LX+ | Carnival ...
...,...,...,...,...
500,telluride,2021,Telluride,Telluride EX AWD | Telluride SX AWD | Tellurid...
501,telluride,2022,Telluride,Telluride EX AWD | Telluride SX AWD | Tellurid...
502,telluride,2023,Telluride,Telluride EX | Telluride SX AWD | Telluride SX...
503,telluride,2024,Telluride,Telluride EX | Telluride SX AWD | Telluride SX...
